# First look at a dataset — Lecture 6

One hour of classroom practice with four synthetic households. One row is one household in 2026; amounts are EUR per month. No real people or population are represented.

Keep `data/raw/households.csv` beside this notebook in the supplied folder structure. Use the existing `research-methods` Python kernel. Run cells from top to bottom with **Shift + Enter**. Predict before running and attempt the practice task before reading its worked answer. The final cells write a derived CSV only to `outputs/`; they preserve the raw input.

These are classroom examples, separate from the graded take-home task. After editing, restart the kernel and run all cells to check that the result does not depend on hidden state.

## The same calculation, now from a file

In [ ]:
income = 1000
consumption = 800
print(income - consumption)

`200` EUR per month. We will obtain these inputs from a CSV, repeat the
calculation for four households, and check the results.

## Open this lesson's folder

In the **terminal**, starting at the course repository root:

```bash
conda activate research-methods
cd codes/lecture-6
jupyter lab
```

Open **`first_dataset.ipynb`**. Keep the `data/raw/` folder beside it.

If you downloaded this lesson on its own, start in that folder and skip the
`cd codes/lecture-6` line. Python code goes in notebook code cells.

## A CSV is a plain-text table

An excerpt showing **three of the six columns**:

```text
household_id,income_eur_month,consumption_eur_month
1,1000,800
2,1200,800
```

- The first line contains column names.
- Each subsequent line contains one household's values.
- Commas separate fields; they are not thousands separators here.

## Define the observation and its units

In [ ]:
household_id = 4
income_eur_month = 800
consumption_eur_month = 850
print(income_eur_month - consumption_eur_month)

`-50` EUR per month: this household consumes more than its current income.

One row is **one synthetic household in 2026**. Household size is persons;
`employed` is an invented `0`/`1` indicator.

## Import pandas once

In [ ]:
import pandas as pd

print("pandas is ready")

```text
pandas is ready
```

**pandas** provides tools for tables. `pd` is the conventional short name used
throughout this notebook.

## Check the path, then read the CSV

In [ ]:
from pathlib import Path

data_path = Path("data/raw/households.csv")
print(data_path.is_file())
households = pd.read_csv(data_path)
print(type(households).__name__)

```text
True
DataFrame
```

If the file check is `False`, check your working folder and filename first.
The path is relative to the folder from which this notebook runs.

## Look at a few rows before calculating

In [ ]:
columns = ["household_id", "income_eur_month",
           "consumption_eur_month"]
preview = households[columns]
print(preview.head(2).to_string(index=False))

```text
 household_id  income_eur_month  consumption_eur_month
            1              1000                    800
            2              1200                    800
```

`head(2)` selects the first two rows for inspection. It does not remove the others.

## Count rows and columns

In [ ]:
print(households.shape)
print(len(households))

```text
(4, 6)
4
```

`.shape` reports **(rows, columns)**. `len(...)` reports the number of rows here.

## Inspect the column names exactly

In [ ]:
for name in households.columns:
    print(name)

```text
household_id
income_eur_month
consumption_eur_month
household_size
employed
survey_year
```

Spelling, underscores, and capitalisation matter when selecting a column.

## Inspect stored types and economic meaning

In [ ]:
columns = ["household_id", "income_eur_month"]
print(households[columns].dtypes.to_string())

```text
household_id        int64
income_eur_month    int64
```

Both are stored as integers, but **an identifier is not an income measurement**.
The storage type alone does not tell you what a column means.

## Select one column

In [ ]:
incomes = households["income_eur_month"]

print(type(incomes).__name__)
print(incomes.to_list())

```text
Series
[1000, 1200, 2000, 800]
```

One selected column is a **Series**. `.to_list()` shows its values as a familiar list.

## Select a table with two columns

In [ ]:
columns = ["household_id", "income_eur_month"]
selected = households[columns]

print(type(selected).__name__)
print(selected.shape)

```text
DataFrame
(4, 2)
```

The selection uses a **list of column names**. It leaves the original table unchanged.

## Select a row by its position

In [ ]:
first_household = households.iloc[0]

print(first_household["household_id"])
print(first_household["income_eur_month"])

```text
1
1000
```

`.iloc[0]` selects the first row by **position**. Its household ID happens to be `1`.

## Your turn · Predict three inspections

1. Predict each output before running the code.
2. Run it and check your predictions.
3. Explain which expression selects rows and which selects columns.

In [ ]:
print(households.shape)
print(households["income_eur_month"].iloc[-1])
print(households[["household_id", "employed"]].shape)

`(4, 6)`, then `800`, then `(4, 2)`.

## Use a Boolean mask to select households

In [ ]:
employed = households["employed"] == 1
print(employed.to_list())
print(households.loc[employed, "household_id"].to_list())

```text
[True, True, True, False]
[1, 2, 3]
```

The comparison produces **one Boolean per row**. `.loc[mask, column]` keeps rows
whose mask is `True` and selects the named column.

## Combine two column comparisons

In [ ]:
mask = ((households["employed"] == 1)
        & (households["income_eur_month"] >= 1200))

print(households.loc[mask, "household_id"].to_list())

```text
[2, 3]
```

Use **`&`** between these pandas comparisons and put each comparison in parentheses.
Python's plain `and` is for the scalar Boolean questions shown earlier.

## Calculate saving for every row

In [ ]:
analysis = households.copy()
analysis["saving_eur_month"] = (
    analysis["income_eur_month"]
    - analysis["consumption_eur_month"]
)

print(analysis["saving_eur_month"].to_list())

```text
[200, 400, 500, -50]
```

The subtraction works **row by row**. The raw CSV and `households` remain unchanged.

## Check a row against a hand calculation

In [ ]:
fourth_household = analysis.iloc[3]
print(fourth_household["saving_eur_month"])

assert fourth_household["saving_eur_month"] == 800 - 850

```text
-50
```

The fourth household's negative saving is an interpretable result.
An assertion should check the intended calculation, not forbid valid outcomes.

## Summarise with the denominator in mind

In [ ]:
print(analysis["saving_eur_month"].sum())
print(analysis["saving_eur_month"].mean())

```text
1050
262.5
```

**Total:** EUR 1,050 per month across four households.\
**Mean:** EUR 262.50 per household per month.

## Check identifiers and missing inputs

In [ ]:
print(households["household_id"].is_unique)
print(households.isna().sum().sum())

assert households["household_id"].is_unique
assert households["income_eur_month"].notna().all()

```text
True
0
```

The fixture has unique IDs and no missing cells. A failed check is a reason to investigate.

## Calculate income per person

In [ ]:
assert (analysis["household_size"] > 0).all()
analysis["income_per_person"] = (
    analysis["income_eur_month"] / analysis["household_size"]
)

print(analysis["income_per_person"].round(2).to_list())

```text
[1000.0, 600.0, 666.67, 400.0]
```

Check the denominator **before dividing**. Rounding here is for the display;
the stored analysis column keeps the calculated precision.

## Your turn · Build and check a small analysis

1. Calculate each household's saving in a copy of the input table.
2. Select the IDs of households with negative saving.
3. Print mean saving and explain its units.

In [ ]:
practice = households.copy()
# TODO: create saving_eur_month
# TODO: select IDs where saving is negative
# TODO: print mean saving

### Worked answer — read after your attempt

In [ ]:
practice["saving_eur_month"] = (
    practice["income_eur_month"] - practice["consumption_eur_month"]
)
negative = practice["saving_eur_month"] < 0
print(practice.loc[negative, "household_id"].to_list())
print(practice["saving_eur_month"].mean())

Negative-saving IDs: `[4]`. Mean saving: **262.5 EUR per household per month**.

## Save a derived result in `outputs/`

In [ ]:
output_path = Path("outputs/household_saving.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

columns = ["household_id", "saving_eur_month"]
analysis[columns].to_csv(output_path, index=False)
print(output_path.is_file())

```text
True
```

`index=False` omits pandas' row index. The household ID remains an explicit column.

## Restart, run all, then check the saved result

In JupyterLab: **Kernel → Restart Kernel and Run All Cells…**

In [ ]:
reloaded = pd.read_csv("outputs/household_saving.csv")
print(reloaded["saving_eur_month"].to_list())
assert reloaded.shape == (4, 2)

```text
[200, 400, 500, -50]
```

A clean run and read-back check complement the earlier hand calculations.

## Exit check · Explain the table and the result

In [ ]:
print(households.shape)
print(analysis["saving_eur_month"].sum())
print(households["household_id"].is_unique)

`(4, 6)`, `1050`, `True`. Explain the observation unit and units of saving.

References: pandas [reading tables](https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html),
[selecting data](https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html),
[derived columns](https://pandas.pydata.org/docs/getting_started/intro_tutorials/05_add_columns.html).